In [6]:
# -*- coding: utf-8 -*-
"""
Q94 4개 항목(도덕성/전문성/사회적영향력/사회기여도) 간 다중공선성 검증

VIF(Variance Inflation Factor)가 높으면(통상 10 이상, 엄격하게는 5 이상)
독립변수끼리 너무 겹쳐서 회귀계수(beta) 해석이 불안정해질 수 있음.
지금까지 회귀분석에서 이 4개를 동시투입한 게 타당했는지 확인.
"""

import pandas as pd
from statsmodels.stats.outliers_influence import variance_inflation_factor
import statsmodels.api as sm

pd.set_option("display.max_columns", None)

DATA_PATH = "2. 2025 언론수용자 조사_최종데이터.xlsx"
df = pd.read_excel(DATA_PATH, sheet_name=0)

predictors = {
    "Q94_1": "도덕성",
    "Q94_2": "전문성",
    "Q94_3": "사회적영향력",
    "Q94_4": "사회기여도",
}
x_cols = list(predictors.keys())

X = df[x_cols].dropna().copy()
X_with_const = sm.add_constant(X)

print("=" * 50)
print("VIF (분산팽창계수) 계산 결과")
print("=" * 50)

vif_rows = []
for i, col in enumerate(X_with_const.columns):
    if col == "const":
        continue
    vif = variance_inflation_factor(X_with_const.values, i)
    vif_rows.append({"변수": predictors[col], "VIF": round(vif, 3)})

vif_df = pd.DataFrame(vif_rows)
print(vif_df.to_string(index=False))

print("\n[참고 기준]")
print("- VIF < 5   : 다중공선성 문제 없음")
print("- 5 <= VIF < 10 : 주의가 필요하나 심각하지 않음")
print("- VIF >= 10 : 다중공선성 심각, 회귀계수 해석에 유의해야 함")

VIF (분산팽창계수) 계산 결과
    변수   VIF
   도덕성 1.233
   전문성 1.319
사회적영향력 1.316
 사회기여도 1.441

[참고 기준]
- VIF < 5   : 다중공선성 문제 없음
- 5 <= VIF < 10 : 주의가 필요하나 심각하지 않음
- VIF >= 10 : 다중공선성 심각, 회귀계수 해석에 유의해야 함


In [9]:
# -*- coding: utf-8 -*-
"""
[1단계] 도덕성 저평가가 신뢰도/언론문제 심각성에 미치는 영향
- 회귀분석 통합본 -

구성:
  (A) 도덕성-신뢰도 연결: 신뢰도 지표 2개를 Q94 4항목으로 회귀
  (B) 도덕성-문제심각성 연결: Q91 8개 항목을 Q94 4항목으로 회귀
  (C) (A)+(B) 종합 요약표: 10개 종속변수 중 도덕성의 상대적 순위

공통 원칙:
  - 모든 변수는 표준화(z-score)하여 표준화계수(beta)로 상대적 크기 비교
  - Q94_1(도덕성)/Q94_2(전문성)/Q94_3(사회적영향력)/Q94_4(사회기여도)를
    동시에 투입해 서로를 통제한 상태에서의 순수 기여도를 봄
  - 인과관계 증명이 아니라 "상대적 연관 강도" 비교임에 유의
"""

import pandas as pd
import statsmodels.api as sm

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 200)

DATA_PATH = "2. 2025 언론수용자 조사_최종데이터.xlsx"
df = pd.read_excel(DATA_PATH, sheet_name=0)
N = len(df)
print(f"전체 응답자 수: {N}\n")

# 공통 독립변수: Q94 4개 항목
predictors = {
    "Q94_1": "도덕성",
    "Q94_2": "전문성",
    "Q94_3": "사회적영향력",
    "Q94_4": "사회기여도",
}
x_cols = list(predictors.keys())


def run_regression(y_col, y_label, controls=None, use_weight=True):
    """Q94 4항목(+선택적 통제변수)으로 y_col을 표준화 가중회귀(WLS). model과 요약행 반환"""
    cols = [y_col] + x_cols + (controls or []) + ["WT"]
    d = df[cols].dropna().copy()

    z_cols = [y_col] + x_cols + (controls or [])
    z = (d[z_cols] - d[z_cols].mean()) / d[z_cols].std()  # 표준화는 가중치 반영 안 함(계수해석 단순화 목적)

    X = sm.add_constant(z[x_cols + (controls or [])])
    y = z[y_col]

    if use_weight:
        model = sm.WLS(y, X, weights=d["WT"]).fit()
    else:
        model = sm.OLS(y, X).fit()

    betas = model.params.drop("const")
    pvals = model.pvalues.drop("const")

    # Q94 4항목 중 도덕성의 절대영향력 순위
    q94_betas = betas[x_cols].abs().sort_values(ascending=False)
    moral_rank = list(q94_betas.index).index("Q94_1") + 1

    return model, betas, pvals, moral_rank, len(d)


def print_block(y_label, y_type, model, betas, pvals, moral_rank):
    print(f"\n[{y_label}] ({y_type}, n={int(model.nobs)}, R²={model.rsquared:.3f})")
    for col, label in predictors.items():
        p = pvals[col]
        sig = "***" if p < .001 else ("**" if p < .01 else ("*" if p < .05 else ""))
        print(f"  {label:8s}: beta={betas[col]:+.3f} {sig}, p={p:.3f} {sig}")
    print(f"  -> 도덕성 순위(Q94 4개 항목 중): {moral_rank}위")


# ======================================================================
# (A) 도덕성 - 신뢰도 연결
# ======================================================================
print("=" * 90)
print("(A) 도덕성-신뢰도 연결 회귀분석")
print("=" * 90)

trust_outcomes = {
    "Q85_1": ("언론 공정성 인식", "긍정지표"),
    "Q86_1": ("정확한 정보제공 수행도", "긍정지표"),
    "Q85_3": ("언론 정확성 인식", "긍정지표"),
    "Q87_1": ("뉴스전반 신뢰도", "긍정지표"),
    "Q95_5": ("언론인 신뢰도(직업군비교)", "긍정지표"),
    "Q86_7": ("사회적약자 대변 수행도", "긍정지표"),
    "Q87_2": ("실이용뉴스 신뢰도", "긍정지표"),
    "Q96": ("사회전반 신뢰도", "긍정지표"),
}

summary_rows = []

for y_col, (y_label, y_type) in trust_outcomes.items():
    model, betas, pvals, moral_rank, n = run_regression(y_col, y_label)
    print_block(y_label, y_type, model, betas, pvals, moral_rank)
    row = {"종속변수": y_label, "유형": y_type, "n": n, "R²": round(model.rsquared, 3)}
    for col, label in predictors.items():
        row[f"{label}_beta"] = round(betas[col], 3)
    row["도덕성_순위"] = moral_rank
    summary_rows.append(row)


# ======================================================================
# (B) 도덕성 - 언론문제 심각성 연결
# ======================================================================
print("\n" + "=" * 90)
print("(B) 도덕성-언론문제 심각성 연결 회귀분석")
print("=" * 90)

problem_outcomes = {
    "Q91_1": ("오보 심각성", "부정지표"),
    "Q91_2": ("낚시성기사 심각성", "부정지표"),
    "Q91_3": ("어뷰징기사 심각성", "부정지표"),
    "Q91_4": ("편파적기사 심각성", "부정지표"),
    "Q91_5": ("광고성기사 심각성", "부정지표"),
    "Q91_6": ("자사이기주의기사 심각성", "부정지표"),
    "Q91_7": ("받아쓰기식기사 심각성", "부정지표"),
    "Q91_8": ("가짜뉴스 심각성", "부정지표"),
}

for y_col, (y_label, y_type) in problem_outcomes.items():
    model, betas, pvals, moral_rank, n = run_regression(y_col, y_label)
    print_block(y_label, y_type, model, betas, pvals, moral_rank)
    row = {"종속변수": y_label, "유형": y_type, "n": n, "R²": round(model.rsquared, 3)}
    for col, label in predictors.items():
        row[f"{label}_beta"] = round(betas[col], 3)
    row["도덕성_순위"] = moral_rank
    summary_rows.append(row)


# ======================================================================
# (C) 종합 요약표
# ======================================================================
print("\n" + "=" * 90)
print("(C) 종합 요약: 10개 종속변수 중 도덕성의 상대적 영향력 순위")
print("=" * 90)

summary_df = pd.DataFrame(summary_rows)
display_cols = ["종속변수", "유형", "R²", "도덕성_beta", "전문성_beta",
                "사회적영향력_beta", "사회기여도_beta", "도덕성_순위"]
print(summary_df[display_cols].to_string(index=False))

전체 응답자 수: 6000

(A) 도덕성-신뢰도 연결 회귀분석

[언론 공정성 인식] (긍정지표, n=6000, R²=0.299)
  도덕성     : beta=+0.475 ***, p=0.000 ***
  전문성     : beta=+0.121 ***, p=0.000 ***
  사회적영향력  : beta=-0.015 , p=0.207 
  사회기여도   : beta=+0.034 **, p=0.008 **
  -> 도덕성 순위(Q94 4개 항목 중): 1위

[정확한 정보제공 수행도] (긍정지표, n=6000, R²=0.240)
  도덕성     : beta=+0.291 ***, p=0.000 ***
  전문성     : beta=+0.154 ***, p=0.000 ***
  사회적영향력  : beta=+0.084 ***, p=0.000 ***
  사회기여도   : beta=+0.132 ***, p=0.000 ***
  -> 도덕성 순위(Q94 4개 항목 중): 1위

[언론 정확성 인식] (긍정지표, n=6000, R²=0.202)
  도덕성     : beta=+0.276 ***, p=0.000 ***
  전문성     : beta=+0.147 ***, p=0.000 ***
  사회적영향력  : beta=+0.085 ***, p=0.000 ***
  사회기여도   : beta=+0.095 ***, p=0.000 ***
  -> 도덕성 순위(Q94 4개 항목 중): 1위

[뉴스전반 신뢰도] (긍정지표, n=6000, R²=0.199)
  도덕성     : beta=+0.292 ***, p=0.000 ***
  전문성     : beta=+0.100 ***, p=0.000 ***
  사회적영향력  : beta=+0.041 **, p=0.001 **
  사회기여도   : beta=+0.137 ***, p=0.000 ***
  -> 도덕성 순위(Q94 4개 항목 중): 1위

[언론인 신뢰도(직업군비교)] (긍정지표, n=6000, R²=0.193)
  도덕성